<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/keypoint_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')
!pip install ultralytics

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.6 MB/s eta 0:00:00


In [5]:
import zipfile
import os
from google.colab import drive

drive.mount('/content/drive')

root = "/content/drive/MyDrive/shoplifting"
extract_to = "/content/shoplifting_data"

os.makedirs(extract_to, exist_ok=True)

for file in os.listdir(root):
    if file.endswith('.zip'):
        zip_path = os.path.join(root, file)
        folder_name = file.replace('.zip', '').replace('.', '_')
        out_path = os.path.join(extract_to, folder_name)
        if not os.path.exists(out_path):
            print(f"Extracting {file}...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(out_path)
            print(f"  Done → {out_path}")
        else:
            print(f"Already extracted: {file}")

print("\nAll ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting shoplifting.zip...
  Done → /content/shoplifting_data/shoplifting
Extracting shoplifting_1.zip...
  Done → /content/shoplifting_data/shoplifting_1
Extracting shoplifting.v1i.yolov11.zip...
  Done → /content/shoplifting_data/shoplifting_v1i_yolov11
Extracting Shoplifting.v2i.yolov11.zip...
  Done → /content/shoplifting_data/Shoplifting_v2i_yolov11

All ready.


#keypoint extraction


In [6]:
import cv2
import numpy as np
import os
import json
import shutil
from ultralytics import YOLO
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# ─── STEP 1: Extract zips ───
root = "/content/drive/MyDrive/shoplifting"
extract_to = "/content/shoplifting_data"
os.makedirs(extract_to, exist_ok=True)

for file in os.listdir(root):
    if file.endswith('.zip'):
        zip_path = os.path.join(root, file)
        folder_name = file.replace('.zip', '').replace('.', '_')
        out_path = os.path.join(extract_to, folder_name)
        if not os.path.exists(out_path):
            print(f"Extracting {file}...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(out_path)
            print(f"  Done → {out_path}")
        else:
            print(f"Already extracted: {file}")

# ─── STEP 2: Check if keypoints already done ───
output_dir = "/content/keypoints"
drive_output = "/content/drive/MyDrive/shoplifting/keypoints"
os.makedirs(output_dir, exist_ok=True)

if os.path.exists(drive_output) and len(os.listdir(drive_output)) > 100:
    print("\nKeypoints already exist in Drive. Loading...")
    shutil.copytree(drive_output, output_dir, dirs_exist_ok=True)
    print(f"Loaded {len(os.listdir(output_dir))} files. Skipping extraction.")

else:
    print("\nStarting keypoint extraction...")

    # ─── STEP 3: Extract keypoints ───
    model = YOLO("yolo11n-pose.pt")

    dataset_paths = [
        ("/content/shoplifting_data/shoplifting/shoplifting", 1),
        ("/content/shoplifting_data/shoplifting/normal", 0),
        ("/content/shoplifting_data/shoplifting_1/Shop DataSet/shop lifters", 1),
        ("/content/shoplifting_data/shoplifting_1/Shop DataSet/non shop lifters", 0),
    ]

    SAMPLE_EVERY = 3
    MAX_FRAMES = 50
    total = 0
    skipped = 0

    def extract_keypoints(video_path):
        cap = cv2.VideoCapture(video_path)
        keypoint_sequence = []
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % SAMPLE_EVERY == 0:
                results = model(frame, verbose=False)
                if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
                    kps = results[0].keypoints.xy[0].cpu().numpy()
                    keypoint_sequence.append(kps.flatten().tolist())
                else:
                    keypoint_sequence.append([0] * 34)
                if len(keypoint_sequence) >= MAX_FRAMES:
                    break
            frame_idx += 1
        cap.release()
        return keypoint_sequence

    for folder_path, label in dataset_paths:
        files = [f for f in os.listdir(folder_path) if f.endswith('.mp4')]
        print(f"\nProcessing: {folder_path} — {len(files)} clips")

        for filename in files:
            video_path = os.path.join(folder_path, filename)
            sequence = extract_keypoints(video_path)

            if len(sequence) < 10:
                skipped += 1
                continue

            while len(sequence) < MAX_FRAMES:
                sequence.append([0] * 34)
            sequence = sequence[:MAX_FRAMES]

            output = {
                "file": filename,
                "label": label,
                "keypoints": sequence
            }

            out_path = os.path.join(output_dir, f"{label}_{total}.json")
            with open(out_path, 'w') as f:
                json.dump(output, f)

            total += 1
            if total % 50 == 0:
                print(f"  Processed {total} clips...")

    print(f"\nExtraction done. {total} clips. {skipped} skipped.")

    # ─── STEP 4: Save to Drive ───
    print("\nSaving keypoints to Drive...")
    if os.path.exists(drive_output):
        shutil.rmtree(drive_output)
    shutil.copytree(output_dir, drive_output)
    print(f"Saved to Drive → {drive_output}")
    print(f"Total files saved: {len(os.listdir(drive_output))}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already extracted: shoplifting.zip
Already extracted: shoplifting_1.zip
Already extracted: shoplifting.v1i.yolov11.zip
Already extracted: Shoplifting.v2i.yolov11.zip

Starting keypoint extraction...

Processing: /content/shoplifting_data/shoplifting/shoplifting — 92 clips
  Processed 50 clips...

Processing: /content/shoplifting_data/shoplifting/normal — 90 clips
  Processed 100 clips...
  Processed 150 clips...

Processing: /content/shoplifting_data/shoplifting_1/Shop DataSet/shop lifters — 324 clips
  Processed 200 clips...
  Processed 250 clips...
  Processed 300 clips...
  Processed 350 clips...
  Processed 400 clips...
  Processed 450 clips...
  Processed 500 clips...

Processing: /content/shoplifting_data/shoplifting_1/Shop DataSet/non shop lifters — 531 clips
  Processed 550 clips...
  Processed 600 clips...
  Processed 650 clips...
  Processed 700 cli